In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
random.seed(42)
import gensim
from sklearn import metrics
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from collections import Counter
from tqdm import tqdm

In [6]:
data = pd.read_csv('CSV table data')
"""
The table contains the following data:
1. antibody: HCDR1-HCDR3, FRH1-FRH3, LCDR1-LCDR3, FRL1-FRL3
2. antigen
3. lebl
"""

In [11]:
AA_scores = {'A':8.1,'R':10.5,'N':11.6,'D':13.0,'C':5.5,'E':12.3,'Q':10.5,'G':9.0,'H':10.4,'I':5.2,
                     'L':4.9,'K':11.3,'M':5.7,'F':5.2,'P':8.0,'S':9.2,'T':8.6,'W':5.4,'Y':6.2,'V':5.9} # grantham scores

# AA_scores = {'A': 5.33, 'R': 4.18, 'D': 3.59, 'N': 3.59, 'C': 7.93, 'Q': 3.87, 'E': 3.65, 'G': 4.48, 'H': 5.1, 'I': 8.83,
#                      'L': 8.47, 'K': 2.95, 'M': 8.95, 'F': 9.03, 'P': 3.87, 'S': 4.09, 'T': 4.49, 'W': 7.66, 'Y': 5.89, 'V': 7.63} #Miyazawa S Hydrophobicity scale
AAs = list(AA_scores.keys())
aa_score_dict = {}
for i in range(len(AAs)): # create all pairs of AAs
    for j in range(len(AAs)-i):
        AA_pair = AAs[i]+AAs[j+i]
        AA_pair_score = round(abs(AA_scores[AAs[i]]-AA_scores[AAs[j+i]])) # take rounded, absolute value of the difference of scores
        aa_score_dict[AA_pair] = AA_pair_score # forward
        aa_score_dict[AA_pair[::-1]] = AA_pair_score # and reverse

In [134]:
def get_window_encodings(df, padding_score=9): # takes df (epitope/receptor sequences) and window (size of epitope).
    """
    Takes a pandas dataframe where each row represents a protein-protein/peptide-protein interaction.

    Customization includes setting the interactor protein and the peptide window. In the pMHC context, the epitope defines the peptide window. In the missense mutation pertubation context, the window_k parameter defines the size of the window and the mutation defines the position. Additionally, the scale used to calculate the score can be altered. If the scale is changed the padding_score may need to be adjusted.

    The function returns a list of score encodings strings that each represent a PPI. The ends of the encodings include padding from the sliding window process. These encodings will be broken into k-mers for the embedding model.

    data1 : antibody: HCDR1-HCDR3, FRH1-FRH3, LCDR1-LCDR3, FRL1-FRL3
    data2: antigen
    """
    total_encodings = [] # final list of encodings

    for i in (df.index): # iterate through all pairs
        seq1 = df['data1'].iloc[i]
        seq2 = df['data2'].iloc[i]

        if len(seq1) <= len(seq2):
            mut_window = seq1
            interactor = seq2
        else:
            mut_window = seq2
            interactor = seq1

        PPI_encoding = '' # for each PPI, dealing with strings
        its = 0 # for sliding window
        for j in range(len(interactor)): # sliding mutant window across entire interactor

            window_scores = ''
            for k in range(len(mut_window)): # at each positon of the interactor, align epitope window and find the score differences
                try: # no directionality
                    pair = mut_window[k]+interactor[k+its]
                    score = aa_score_dict[pair]

                except: # if not a pair, it is padding (have reached the end of the interactor)
                    pair = None
                    score = padding_score # padding
                window_scores = window_scores + str(score) # string per epitope window

            its +=1 # sliding down to next position on the interactor
            PPI_encoding = PPI_encoding + str(window_scores) # final string per interaction

        total_encodings.append(PPI_encoding) # all strings for all interactions

    return total_encodings

In [156]:
data = data.sample(frac=1).reset_index(drop=True)

In [135]:
window_encodings = get_window_encodings(data, padding_score=9) # still encode all and get k-mers

In [136]:
def get_kmers_str(encoding_scores, k=7, padding_score=9):
    """
    Takes the encoding scores from get_window_encodings().

    Customization includes setting size of the kmers (k), a shuffle option, and the integer defining the padding score.

    This function returns a list of lists of overlapping k-mers of specified size k, removing k-mers of only padding. Each list of k-mers are specific to each of the PPIs. This output is compatible with gensims
    """

    padding = {str(padding_score)}
    for i in range(k):
        padding.add(str(padding_score)*(i+1))
    kmers = []
    for ppi_score in encoding_scores:
        int_kmers = []
        for j in range(len(ppi_score)): # keep padding in k-mers?
            kmer = ppi_score[j:j+k]
            if kmer in padding:
                pass
            else:
                int_kmers.append(kmer) # overlapping k-mers
        kmers.append(int_kmers)
    return kmers

In [137]:
kmers = get_kmers_str(window_encodings,k=7, padding_score=9)

In [138]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=kmers,
    vector_size=64,   # embedding维度（推荐64~256）
    window=5,
    min_count=1,
    workers=8,
    sg=1  # skip-gram（推荐）
)

In [139]:
import numpy as np

def get_seq_embedding(seq, model):
    vecs = []
    for token in seq:
        if token in model.wv:
            vecs.append(model.wv[token])

    if len(vecs) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vecs, axis=0)

In [140]:
feature = np.array([get_seq_embedding(seq, w2v_model) for seq in kmers])

In [3]:
feature

In [142]:
data['HCDR1_w2v'] = feature.tolist()   ##The interaction embedding between the stored CDR (FR) and the antibody sequence.

In [5]:
data

In [144]:
data.to_csv('Save file!', index=False) # Store data in CSV format.